In [11]:
from pathlib import Path
from suite2p.default_settings import default_settings
from suite2p import run_s2p
import tifffile

ModuleNotFoundError: No module named 'suite2p.default_settings'

In [8]:
TIFF_FOLDER = Path('/Volumes/rkc_ramirezlab/Home/suthardr/Projects/2photon_menace/FriendsCohort/friends_cohort_june2026/Day2-TFC/friends_chandler_day2_06162026_extracted')
OUT_ROOT = Path('/Volumes/rkc_ramirezlab/Home/suthardr/Projects/2photon_menace/FriendsCohort/friends_cohort_june2026/Day2-TFC/friends_chandler_day2_06162026_extracted/suite2p_outputs')


FRAME_RATE = 11.756  # change this to your true imaging frame rate

for tif_path in tiff_files:
    print(f'\nRunning Suite2p on {tif_path.name}')

    with tifffile.TiffFile(tif_path) as tif:
        arr_shape = tif.series[0].shape

    if len(arr_shape) == 3:
        n_frames, Ly, Lx = arr_shape
    elif len(arr_shape) == 2:
        Ly, Lx = arr_shape
        n_frames = 1
    else:
        raise ValueError(f'Unexpected TIFF shape for {tif_path.name}: {arr_shape}')

    save_dir = OUT_ROOT / f'{tif_path.stem}_suite2p'
    save_dir.mkdir(parents=True, exist_ok=True)

    # db = recording-specific / file-location parameters
    db = {
        'data_path': [str(tif_path.parent)],
        'file_list': [tif_path.name],       # note: renamed from tiff_list -> file_list
        'save_path0': str(save_dir),
        'input_format': 'tif',              # note: 'tif', not 'tiff'
        'nplanes': 1,
        'nchannels': 1,
    }

    # settings = pipeline behavior, nested by category
    settings = default_settings()
    settings['fs'] = FRAME_RATE
    settings['run']['do_registration'] = 1
    settings['run']['do_detection'] = True
    settings['run']['do_deconvolution'] = True
    settings['io']['save_mat'] = False

    run_s2p(db=db, settings=settings)

    print(f'Finished: {save_dir}')


Running Suite2p on channel_green.tif


TypeError: run_s2p() got an unexpected keyword argument 'ops'

# Femtonics → Suite2p batch runner

Takes the output of `Femtonics_Data_export.ipynb` (folders containing
`channel_green.tif`, `channel_red.tif`, and `metadata.json`) and runs
Suite2p on both channels separately.

Configured for suite2p's current `db` + nested `settings` API, and for
Apple Silicon (`torch_device='mps'`).

Point `INPUT_PATH` at either:
- a single `*_extracted` folder, **or**
- a parent folder containing multiple `*_extracted` folders (batch mode)

In [12]:
from pathlib import Path
import json
import tifffile
from suite2p import run_s2p, default_settings, default_db

In [13]:
def verify_tiff_matches_metadata(tif_path, expected_frames):
    """Sanity check that the on-disk tiff wasn't truncated during export
    (the ImageJ-format writer can silently drop frames past its size ceiling)."""
    with tifffile.TiffFile(tif_path) as tif:
        shape = tif.series[0].shape
    n_frames_on_disk = shape[0] if len(shape) == 3 else 1
    if n_frames_on_disk != expected_frames:
        print(f'  ⚠ FRAME COUNT MISMATCH for {tif_path.name}: '
              f'metadata says {expected_frames}, tiff on disk has {n_frames_on_disk}')
        print(f'    This file may have been truncated during export.')
        return False
    return True


def run_suite2p_on_channel(tif_path, fs, out_root):
    save_dir = out_root / f'{tif_path.stem}_suite2p'
    save_dir.mkdir(parents=True, exist_ok=True)

    # db: recording / file-location parameters
    db = default_db()
    db['data_path']    = [str(tif_path.parent)]
    db['file_list']    = [tif_path.name]
    db['save_path0']   = str(save_dir)
    db['input_format'] = 'tif'
    db['nplanes']      = 1
    db['nchannels']    = 1

    # settings: pipeline behavior
    settings = default_settings()
    settings['fs'] = fs
    settings['torch_device'] = 'mps'  # Apple Silicon GPU (falls back to CPU if unavailable)
    settings['run']['do_registration']   = 1
    settings['run']['do_detection']      = True
    settings['run']['do_deconvolution']  = True
    settings['io']['save_mat'] = False

    print(f'  Running suite2p on {tif_path.name}  (fs={fs:.3f} Hz, device=mps)')
    run_s2p(db=db, settings=settings)
    print(f'  ✓ Finished: {save_dir}')


def process_extracted_folder(folder):
    print(f'\n{"─"*70}')
    print(f'  {folder.name}')
    print(f'{"─"*70}')

    meta_path = folder / 'metadata.json'
    if not meta_path.exists():
        print(f'  ⚠ no metadata.json found — skipping (need this for frame rate)')
        return

    with open(meta_path) as f:
        meta = json.load(f)
    fs = meta['munit']['true_frame_rate_hz']
    expected_frames = meta['munit']['true_n_frames']

    out_root = folder / 'suite2p_outputs'
    out_root.mkdir(exist_ok=True)

    for color in ['green', 'red']:
        tif_path = folder / f'channel_{color}.tif'
        if not tif_path.exists():
            print(f'  skipping channel_{color}.tif — not found')
            continue

        ok = verify_tiff_matches_metadata(tif_path, expected_frames)
        if not ok:
            print(f'  ⚠ Skipping suite2p run for {tif_path.name} until this is resolved.')
            print(f'    (Re-export with bigtiff=True in the export notebook, or confirm the mismatch is expected.)')
            continue

        run_suite2p_on_channel(tif_path, fs, out_root)

In [14]:
# ── set this to a single *_extracted folder OR a parent folder containing several ──
INPUT_PATH = Path('/Volumes/rkc_ramirezlab/Home/suthardr/Projects/2photon_menace/FriendsCohort/friends_cohort_june2026/Day2-TFC/friends_chandler_day2_06162026_extracted')

if (INPUT_PATH / 'metadata.json').exists():
    folders_to_process = [INPUT_PATH]
else:
    folders_to_process = sorted(INPUT_PATH.glob('*_extracted'))

if not folders_to_process:
    print(f'No extracted folders found at {INPUT_PATH}')
else:
    print(f'Found {len(folders_to_process)} folder(s) to process')

    failed = []
    for folder in folders_to_process:
        try:
            process_extracted_folder(folder)
        except Exception as e:
            print(f'  ✗ FAILED: {e}')
            failed.append((folder.name, str(e)))

    print(f'\n{"═"*70}')
    print(f'Done. {len(folders_to_process) - len(failed)}/{len(folders_to_process)} folder(s) succeeded.')
    if failed:
        print('Failed:')
        for name, err in failed:
            print(f'  {name}: {err}')

Found 1 folder(s) to process

──────────────────────────────────────────────────────────────────────
  friends_chandler_day2_06162026_extracted
──────────────────────────────────────────────────────────────────────
  Running suite2p on channel_green.tif  (fs=11.755 Hz, device=mps)
  ✗ FAILED: number of frames should be at least 50

══════════════════════════════════════════════════════════════════════
Done. 0/1 folder(s) succeeded.
Failed:
  friends_chandler_day2_06162026_extracted: number of frames should be at least 50


In [16]:
import tifffile
with tifffile.TiffFile('/Volumes/rkc_ramirezlab/Home/suthardr/Projects/2photon_menace/FriendsCohort/friends_cohort_june2026/Day2-TFC/friends_chandler_day2_06162026_extracted/channel_green.tif') as tif:
    print(tif.series[0].shape)

(10815, 656, 656)
